# Install Dependencies

In [1]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.12.12 environment at: /Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/.venv
Resolved 70 packages in 678ms                                        
⠙ Preparing packages... (0/19)                                                  
⠙ Preparing packages... (0/19)------------------     0 B/1.10 MiB            
⠙ Preparing packages... (0/19)------------------     0 B/1.10 MiB            
orjson               ------------------------------     0 B/125.75 KiB
⠙ Preparing packages... (0/19)------------------     0 B/1.10 MiB            
orjson               ------------------------------     0 B/125.75 KiB
⠙ Preparing packages... (0/19)------------------     0 B/1.10 MiB            
orjson               ------------------------------     0 B/125.75 KiB
⠙ Preparing packages... (0/19)------------------     0 B/1.10 MiB            
orjson               ------------------------------     0 B/125.75 KiB
regex                ------------------------------     0 B/282.

In [3]:
import os

from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Data Ingestion

In [4]:
from langchain_community.document_loaders import TextLoader

In [8]:
loader = TextLoader("/Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/data/artemis2.txt", encoding="utf8")
documents = loader.load()

In [9]:
documents[0].page_content[:500]

'NASA’s Artemis II mission stands as one of the most important milestones in modern space exploration because it marked humanity’s return to the Moon’s vicinity with astronauts for the first time since the Apollo era. As of April 12, 2026, Artemis II has already completed its flight: it launched on April 1, 2026, flew a crewed lunar flyby mission, and splashed down on April 10, 2026, after a mission lasting 9 days, 1 hour, and 32 minutes. More than just a symbolic return, Artemis II was the first'

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [12]:
text_chunks=text_splitter.split_documents(documents)

In [14]:
text_chunks

[Document(metadata={'source': '/Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/data/artemis2.txt'}, page_content='NASA’s Artemis II mission stands as one of the most important milestones in modern space exploration because it marked humanity’s return to the Moon’s vicinity with astronauts for the first time since'),
 Document(metadata={'source': '/Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/data/artemis2.txt'}, page_content='first time since the Apollo era. As of April 12, 2026, Artemis II has already completed its flight: it launched on April 1, 2026, flew a crewed lunar flyby mission, and splashed down on April 10,'),
 Document(metadata={'source': '/Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/data/artemis2.txt'}, page_content='down on April 10, 2026, after a mission lasting 9 days, 1 hour, and 32 minutes. More than just a symbolic return, Artemis II was the first crewed Artemis mission and the first full end-to-end tes

In [15]:
! uv pip install faiss-cpu

Using Python 3.12.12 environment at: /Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/.venv
Resolved 3 packages in 184ms                                         
Installed 1 package in 10ms                                 
 + faiss-cpu==1.13.2


In [17]:
from langchain_classic.embeddings import OpenAIEmbeddings
from langchain_classic.vectorstores import FAISS

In [18]:
embeddings=OpenAIEmbeddings()

/var/folders/7f/ckzvnvjx6zgfg_hdwnc3cb200000gn/T/ipykernel_11541/2276573386.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  embeddings=OpenAIEmbeddings()


In [ ]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [20]:
vectorstore

In [21]:
retriever=vectorstore.as_retriever()

In [22]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
that the full mission profile—from launch to deep-space transit to Earth return—can be executed with astronauts aboard.
--------------------------------------------------
Document 2:
Artemis II also carried science investigations intended to improve understanding of the human body in deep space. One of the best-known examples was AVATAR, which used organ-on-a-chip devices
--------------------------------------------------
Document 3:
[5]: https://www.nasa.gov/missions/nasa-answers-your-most-pressing-artemis-ii-questions/ "NASA Answers Your Most Pressing Artemis II Questions - NASA"
--------------------------------------------------
Document 4:
A major reason Artemis II captured worldwide attention was its crew. The spacecraft carried four astronauts: Reid Wiseman as commander, Victor Glover as pilot, Christina Koch as mission specialist,
--------------------------------------------------


In [23]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [24]:
prompt=ChatPromptTemplate.from_template(template)

In [25]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [28]:
from langchain_core.output_parsers import StrOutputParser

In [29]:
output_parser=StrOutputParser()

In [33]:
! uv pip install langchain-openai

Using Python 3.12.12 environment at: /Users/hdksrma/Desktop/Courses/projects/MultiDocChat-Production-RAG/.venv
Resolved 34 packages in 380ms                                        
Installed 1 package in 9ms.1.12                             
 + langchain-openai==1.1.12


In [34]:
from langchain_openai import ChatOpenAI

llm_model=ChatOpenAI(model_name="gpt-4o-mini")

In [36]:
from langchain_core.runnables import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [38]:
rag_chain.invoke("tell me about Artemis")

"Artemis is a human spaceflight program aimed at lunar exploration and eventually deeper space missions. Artemis II marked a significant step by transitioning the program from a conceptual promise to a practical human spaceflight campaign. This mission carried a crew of four astronauts, including Commander Reid Wiseman, Pilot Victor Glover, and Mission Specialist Christina Koch. Artemis I demonstrated the capabilities of the Orion spacecraft and Space Launch System (SLS), setting the stage for Artemis II. The mission not only focused on exploration but also included scientific investigations to enhance understanding of the human body's response to deep space conditions. A notable investigation was the AVATAR project, which utilized organ-on-a-chip technology. Overall, Artemis II is a pivotal moment for the future of lunar exploration and human space travel."